In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
## 데이터셋, 데이터로더 관련 모듈
from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk
from nltk.tokenize import word_tokenize

In [2]:
FILE_DIR = './data/'
FILE_PATH = FILE_DIR+'IMDb_Reviews.csv'


In [3]:
import string

STOPWORDS = stopwords.words('english')
TOKENIZER = get_tokenizer('basic_english')
PUNC = string.punctuation
PUNC = PUNC+''.join([str(x) for x in range(10)])

In [4]:
dataDF= pd.read_csv(FILE_PATH)
texts = dataDF['review'].tolist()
labels = dataDF['sentiment'].tolist()
dataDF.head()

,review,sentiment
0,My family and I normally do not watch local mo...,1
1,"Believe it or not, this was at one time the wo...",0
2,"After some internet surfing, I found the ""Home...",0
3,One of the most unheralded great works of anim...,1
4,"It was the Sixties, and anyone with long hair ...",0


In [5]:
def yield_tokens(data):
    for line in data:
        line = ''.join([x for x in line if x not in PUNC])
        yield word_tokenize(line.lower())


In [6]:
VOCAB = build_vocab_from_iterator(yield_tokens(texts), specials=["<unk>", "<pad>"])
VOCAB.set_default_index(VOCAB["<unk>"])

In [7]:
def encode_texts(data, vocab):
    encoded = []
    for line in data:
        line = ''.join([c for c in line if c not in PUNC])
        tokens = word_tokenize(line.lower())
        token_ids = [vocab[token] for token in tokens]
        encoded.append(torch.tensor(token_ids, dtype=torch.long))
    return encoded

In [8]:
encoded_sequences = encode_texts(texts, VOCAB)

padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=VOCAB["<pad>"])
labels_tensor = torch.tensor(labels, dtype=torch.long)


In [9]:
class customDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    padded_sequences, labels_tensor, test_size=0.2, random_state=42, stratify=labels_tensor
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [16]:
train_dataset = customDataset(X_train, y_train)
val_dataset = customDataset(X_val, y_val)
test_dataset = customDataset(X_test, y_test)

In [20]:
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [19]:
len(train_dataset)

40000

In [21]:
# 예: padded_sequences → BoW 텐서 (VOCAB_SIZE 크기의 벡터)
def sequences_to_bow(sequences, vocab_size):
    bow_vectors = []
    for seq in sequences:
        bow = torch.zeros(vocab_size)
        for idx in seq:
            if idx != VOCAB["<pad>"]:
                bow[idx] += 1
        bow_vectors.append(bow)
    return torch.stack(bow_vectors)

In [22]:
class DNN_BoW(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(DNN_BoW, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out = self.relu(self.fc1(x))    # [batch, hidden_dim]
        out = self.dropout(out)
        out = self.fc2(out)             # [batch, num_classes]
        return out

In [23]:
class RNN_OneHot(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_classes):
        super(RNN_OneHot, self).__init__()
        self.rnn = nn.GRU(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):  # x: [batch, seq_len], 정수 인덱스
        one_hot = F.one_hot(x, num_classes=vocab_size).float()  # [batch, seq_len, vocab_size]
        _, hidden = self.rnn(one_hot)  # hidden: [1, batch, hidden_size]
        out = self.fc(hidden.squeeze(0))  # [batch, num_classes]
        return out